# **Hackathon: Job Coach**

This JobCoach can analyze resumes against job descriptions, identify skill gaps with personalized learning paths, generate targeted interview questions, and create strategic career development plans.


---



This section of code is for setup and instalizaed of required libraries and other packages:


---



In [ ]:
# --- Install Required Libraries ---
!pip install -q --upgrade langchain langchain-community langchain-openai openai langchain-google-genai langchain-anthropic
!pip install -q PyPDF2 python-docx pdfplumber tiktoken
!pip install -q gradio


# --- Import Python Libraries ---
import os
import PyPDF2
import gradio as gr
from google.colab import files, userdata
from IPython.display import display, Markdown, Javascript
import webbrowser

# LangChain Core Components
from langchain.chat_models import ChatOpenAI
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_anthropic import ChatAnthropic
from langchain.prompts import PromptTemplate
from langchain.agents import create_react_agent, AgentExecutor
from langchain.tools import Tool
from langchain_core.output_parsers import StrOutputParser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 19.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.5/64.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 723.4/723.4 kB 31.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 286.3/286.3 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 50.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.5/438.5 kB 25.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 2.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-generativeai 0.8.5 requires google-ai-generativelanguage==0.6.15, but you have google-ai-generativelanguage 0

In [ ]:
# --- API Key Setup ---
openai_key = userdata.get('OPENAI_API_KEY')
gemini_key = userdata.get('Gemini_api_key')
claude_key = userdata.get('CLAUDE_API_KEY')
if openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key
if gemini_key:
    os.environ["GOOGLE_API_KEY"] = gemini_key
if claude_key:
    os.environ["CLAUDE_API_KEY"] = claude_key

In [ ]:
# Set as environment variables for downstream packages (LangChain, etc.)
if openai_key:
    os.environ["OPENAI_API_KEY"] = openai_key
    print("✅ OpenAI API Key loaded from secrets!")
else:
    print("❌ OpenAI API Key not found in secrets. GPT-4 functionality might be limited.")

if gemini_key:
    os.environ["GOOGLE_API_KEY"] = gemini_key
    print("✅ Gemini API Key loaded from secrets!")
else:
    print("❌ Gemini API Key not found in secrets. Gemini functionality might be limited.")

if claude_key:
    os.environ["CLAUDE_API_KEY"] = claude_key
    print("✅ Claude API Key loaded from secrets!")
else:
    print("❌ Claude API Key not found in secrets. Claude functionality might be limited.")

# Check if API keys are set (these checks are good to keep)
if not os.environ.get("OPENAI_API_KEY"):
    print("WARNING: OPENAI_API_KEY environment variable not set. GPT-4 functionality might be limited.")
if not os.environ.get("GOOGLE_API_KEY"):
    print("WARNING: GEMINI_API_KEY environment variable not set. Gemini functionality might be limited.")
if not os.environ.get("CLAUDE_API_KEY"):
    print("WARNING: CLAUDE_API_KEY environment variable not set. Claude functionality might be limited.")

✅ OpenAI API Key loaded from secrets!
✅ Gemini API Key loaded from secrets!
✅ Claude API Key loaded from secrets!




---



## ChatGPT JobCoach  


---



In [ ]:
# --- GPT LLM Initialization ---
llm_gpt4 = ChatOpenAI(model="gpt-4-turbo", temperature=0.2)


# --- Prompt Templates and Chains ---
job_skill_extractor_chain = PromptTemplate.from_template(
    """
    Given this job description:
    {job_text}

    Extract the top technical and soft skills required. List them clearly.
    """
) | llm_gpt4 | StrOutputParser()

resume_skill_extractor_chain = PromptTemplate.from_template(
    """
    Given this resume:
    {resume_text}

    Extract the most prominent technical and soft skills mentioned. List them clearly.
    """
) | llm_gpt4 | StrOutputParser()

skill_gap_analyzer_chain = PromptTemplate.from_template(
    """
    Given this resume:
    {resume_text}

    And this job description:
    {job_text}

    Compare them and identify any specific skill gaps or missing qualifications from the resume perspective,
    relative to the job description's requirements. Be precise.
    """
) | llm_gpt4 | StrOutputParser()

interview_question_generator_chain = PromptTemplate.from_template(
    """
    Create a list of 5-7 targeted behavioral and technical interview questions for the following job role:
    {role}
    """
) | llm_gpt4 | StrOutputParser()

strategic_career_plan_chain = PromptTemplate.from_template(
    """
    Based on the following information, create a strategic career development plan for the user,
    targeting the role of '{job_role}'.

    Resume Skills Summary:
    {resume_skills}

    Job Required Skills Summary:
    {job_skills}

    Identified Skill Gaps:
    {skill_gaps}

    Include short, medium, and long-term goals, recommended resources, and actionable steps.
    """
) | llm_gpt4 | StrOutputParser()

# --- Tools ---
def resume_parser_tool(file_path: str) -> str:
    try:
        with open(file_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            return "".join([page.extract_text() or "" for page in reader.pages]).strip()
    except Exception as e:
        return f"Error reading resume: {e}"

resume_parser = Tool(
    name="ResumeParser",
    func=resume_parser_tool,
    description="Extracts plain text from a PDF resume file given the file path."
)

def extract_skills_from_job_tool(job_text: str) -> str:
    return job_skill_extractor_chain.invoke({"job_text": job_text})

job_skill_extractor_tool_instance = Tool(
    name="JobSkillExtractor",
    func=extract_skills_from_job_tool,
    description="Extracts required skills from a job description."
)

def skill_gap_analysis_tool(input_data: str) -> str:
    separator = "<JOB>"
    if separator not in input_data:
        return "Error: Missing separator '<JOB>'."
    resume_text, job_text = map(str.strip, input_data.split(separator, 1))
    return skill_gap_analyzer_chain.invoke({"resume_text": resume_text, "job_text": job_text})

skill_gap_analyzer_tool_instance = Tool(
    name="SkillGapAnalyzer",
    func=skill_gap_analysis_tool,
    description="Compares resume and job description to find skill gaps."
)

def interview_question_tool_func(role: str) -> str:
    return interview_question_generator_chain.invoke({"role": role})

interview_question_generator_tool_instance = Tool(
    name="InterviewPrepGenerator",
    func=interview_question_tool_func,
    description="Generates interview questions for a given job role."
)

def strategic_career_plan_tool_func(input_data: str) -> str:
    parts = input_data.split("<SEP>")
    if len(parts) != 4:
        return "Error: Expected 4 parts separated by '<SEP>'."
    resume_skills, job_skills, skill_gaps, job_role = map(str.strip, parts)
    return strategic_career_plan_chain.invoke({
        "resume_skills": resume_skills,
        "job_skills": job_skills,
        "skill_gaps": skill_gaps,
        "job_role": job_role
    })

strategic_career_plan_generator_tool_instance = Tool(
    name="StrategicCareerPlanGenerator",
    func=strategic_career_plan_tool_func,
    description="Generates a career development plan from resume and job data."
)

all_tools = [
    resume_parser,
    job_skill_extractor_tool_instance,
    skill_gap_analyzer_tool_instance,
    interview_question_generator_tool_instance,
    strategic_career_plan_generator_tool_instance
]

# --- JobCoach Agent Initialization (GPT-4) ---
gpt4_agent_prompt = PromptTemplate.from_template(
    """
    You are JobCoach, an expert AI career mentor, powered by GPT-4.
    Your goal is to provide comprehensive career guidance.
    You have access to the following tools:
    {tools}

    Use the following format:
    Question: the user's query
    Thought: reason about what to do, what tools to use, and why.
    Action: the action to take, should be one of [{tool_names}]
    Action Input: the input to the action
    Observation: result of the action
    ... (this Thought/Action/Action Input/Observation can repeat N times)
    Thought: I have gathered all the necessary information and can now formulate the final, structured answer.
    Final Answer:
    =============================
    📈 1. Resume vs Job Description Match (GPT-4)
    =============================
    [Your detailed analysis of matching skills and general alignment here.]

    =============================
    🧱 2. Skill Gaps & Personalized Learning Path (GPT-4)
    =============================
    [Your identification of skill gaps and specific learning path recommendations here.]

    =============================
    🎯 3. Targeted Interview Questions (GPT-4)
    =============================
    [Your list of interview questions here.]

    =============================
    📊 4. Strategic Career Development Plan (GPT-4)
    =============================
    [Your comprehensive career development plan here.]

    Begin!

    Question: {input}
    Thought:{agent_scratchpad}
    """
)

gpt4_agent = create_react_agent(
    llm=llm_gpt4,
    tools=all_tools,
    prompt=gpt4_agent_prompt
)

jobcoach_agent_gpt4 = AgentExecutor(
    agent=gpt4_agent,
    tools=all_tools,
    verbose=False,
    handle_parsing_errors=True,
    max_iterations=15,
#    early_stopping_method="generate"
)

# --- Main Execution ---
print("\U0001F4E4 Upload your resume (PDF format)...")
uploaded = files.upload()
file_path = next(iter(uploaded))
resume_text = resume_parser_tool(file_path)

if "Error" in resume_text:
    print(resume_text)
else:
    print("\n\U0001F4C4 Resume parsed successfully!")
    print("\n\U0001F4DD Paste the job description (press Enter twice to finish):")
    job_text = "\n".join(iter(input, ""))
    job_role = input("\n\U0001F50D Job role/title (e.g., 'Data Scientist'): ")

    query = f"""
    Analyze the resume and job description for '{job_role}'.
    Resume:
    {resume_text}

    Job Description:
    {job_text}
    """

    print("\n\U0001F680 GPT-4 JobCoach:")
    display(Markdown(jobcoach_agent_gpt4.invoke({"input": query})["output"]))

print("\n\U0001F4AC Ask career questions (type 'exit' to stop)")
while True:
    user_question = input("\n❓ Your question: ")
    if user_question.lower() in ["exit", "quit"]:
        print("\n👋 Goodbye from JobCoach!")
        break
    display(Markdown(jobcoach_agent_gpt4.invoke({"input": user_question})["output"]))




📤 Upload your resume (PDF format)...


Saving Sample_Data_Analyst_Resume.pdf to Sample_Data_Analyst_Resume (15).pdf

📄 Resume parsed successfully!

📝 Paste the job description (press Enter twice to finish):
We are seeking a skilled Data Engineer to design, build, and maintain scalable data pipelines and infrastructure. You will work closely with data scientists, analysts, and software engineers to ensure efficient data flow and accessibility. Key responsibilities include data ingestion, transformation, storage optimization, and ensuring data quality and integrity.


🔍 Job role/title (e.g., 'Data Scientist'): Data Engineer

🚀 GPT-4 JobCoach:


=============================
📈 1. Resume vs Job Description Match (GPT-4)
=============================
Jane Doe's resume showcases strong foundational skills in data analysis, including proficiency in Python, SQL, Tableau, and Excel. She has a solid background in data visualization, statistical modeling, and has implemented efficiency improvements in her current role. However, when compared to the job description for a Data Engineer, there are notable differences. The role requires specific skills in data pipeline design, data ingestion and transformation, and storage optimization, which are not explicitly mentioned in her resume.

=============================
🧱 2. Skill Gaps & Personalized Learning Path (GPT-4)
=============================
**Skill Gaps Identified:**
- **Data Pipeline Design**
- **Data Ingestion & Transformation**
- **Storage Optimization**
- **Data Quality Assurance**
- **Database Management**
- **Programming in Scala and Java**
- **Big Data Technologies (Hadoop, Spark, Kafka)**
- **Project Management**

**Personalized Learning Path:**
1. **Technical Skills Development:**
   - **Online Courses:** Enroll in courses focusing on data engineering, particularly those covering data pipeline architecture, big data technologies (Hadoop, Spark, Kafka), and programming languages (Scala, Java).
   - **Certifications:** Consider obtaining certifications in big data technologies and advanced database management systems to solidify her credentials.
   - **Practical Projects:** Engage in projects or freelance jobs that involve large-scale data processing and pipeline construction to gain hands-on experience.

2. **Soft Skills Enhancement:**
   - **Project Management Courses:** Take project management courses to understand methodologies and tools essential for managing data-related projects.
   - **Workshops:** Attend workshops on collaboration and communication to enhance these skills, making her a more effective team player in cross-functional environments.

=============================
🎯 3. Targeted Interview Questions (GPT-4)
=============================
1. Describe a time when you had to design and implement a solution to a complex data problem. What approach did you take?
2. How do you ensure the integrity and accuracy of data in your projects?
3. Can you explain a scenario where you had to learn a new technology to complete a project? How did you approach the learning process?
4. Discuss your experience with any big data technologies. What challenges did you face and how did you overcome them?
5. How do you prioritize tasks in a project involving multiple stakeholders?

=============================
📊 4. Strategic Career Development Plan (GPT-4)
=============================
**Short-term Goals (1-2 Years):**
- **Skill Acquisition:** Focus on closing the identified skill gaps through targeted courses and certifications.
- **Networking:** Increase engagement with professional groups and forums related to data engineering to expand her professional network and opportunities.

**Mid-term Goals (3-5 Years):**
- **Role Transition:** Aim to transition into roles that involve more responsibilities in data engineering, such as a lead data engineer position.
- **Leadership Development:** Develop leadership skills by leading small project teams or initiatives within her current organization.

**Long-term Goals (5+ Years):**
- **Industry Impact:** Aspire to become a subject matter expert in data engineering, contributing to industry-wide best practices and innovations.
- **Continuous Learning:** Maintain a commitment to continuous professional development and adaptation to new technologies and methodologies in the field.

By following this structured plan, Jane Doe can effectively prepare for her career transition and future growth in the field of data engineering.


💬 Ask career questions (type 'exit' to stop)

❓ Your question: what career advice can you share?


=============================
📈 1. Resume vs Job Description Match (GPT-4)
=============================
To ensure your resume aligns well with job descriptions, focus on customizing your resume for each job application. Highlight relevant experience, skills, and accomplishments that directly address the requirements listed in the job description. Use keywords from the job description to pass Applicant Tracking Systems (ATS).

=============================
🧱 2. Skill Gaps & Personalized Learning Path (GPT-4)
=============================
Identify skill gaps by comparing your current skills with those required in your desired job roles. You can use online platforms like LinkedIn Learning, Coursera, or Udemy to find courses that address these gaps. Regularly updating your skills is crucial in keeping pace with industry demands.

=============================
🎯 3. Targeted Interview Questions (GPT-4)
=============================
Prepare for interviews by practicing responses to common questions related to your field. Additionally, prepare questions that demonstrate your knowledge of the industry, interest in the role, and your proactive nature. Consider conducting mock interviews with mentors or peers to gain confidence.

=============================
📊 4. Strategic Career Development Plan (GPT-4)
=============================
Develop a strategic career plan that includes short-term and long-term goals. This plan should outline the steps needed to achieve these goals, including education, networking, and skill development. Regularly review and adjust your plan as you progress in your career and as your goals evolve.

These steps provide a foundational approach to managing and advancing your career effectively. For more tailored advice, specific details about your career situation would be necessary.


❓ Your question: exit

👋 Goodbye from JobCoach!


# **Gemini JobCoach**

In [ ]:
# --- LLM Initialization ---
llm_gemini = ChatGoogleGenerativeAI(
              model="gemini-1.5-flash",
              google_api_key=os.environ["GOOGLE_API_KEY"],
              temperature=0.2)

# --- Prompt Templates and Chains ---
job_skill_extractor_chain = PromptTemplate.from_template(
    """
    Given this job description:
    {job_text}

    Extract the top technical and soft skills required. List them clearly.
    """
) | llm_gemini | StrOutputParser()

resume_skill_extractor_chain = PromptTemplate.from_template(
    """
    Given this resume:
    {resume_text}

    Extract the most prominent technical and soft skills mentioned. List them clearly.
    """
) | llm_gemini | StrOutputParser()

skill_gap_analyzer_chain = PromptTemplate.from_template(
    """
    Given this resume:
    {resume_text}

    And this job description:
    {job_text}

    Compare them and identify any specific skill gaps or missing qualifications from the resume perspective,
    relative to the job description's requirements. Be precise.
    """
) | llm_gemini | StrOutputParser()

interview_question_generator_chain = PromptTemplate.from_template(
    """
    Create a list of 5-7 targeted behavioral and technical interview questions for the following job role:
    {role}
    """
) | llm_gemini | StrOutputParser()

strategic_career_plan_chain = PromptTemplate.from_template(
    """
    Based on the following information, create a strategic career development plan for the user,
    targeting the role of '{job_role}'.

    Resume Skills Summary:
    {resume_skills}

    Job Required Skills Summary:
    {job_skills}

    Identified Skill Gaps:
    {skill_gaps}

    Include short, medium, and long-term goals, recommended resources, and actionable steps.
    """
) | llm_gemini | StrOutputParser()

# --- Tools ---
def resume_parser_tool(file_path: str) -> str:
    try:
        with open(file_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            return "".join([page.extract_text() or "" for page in reader.pages]).strip()
    except Exception as e:
        return f"Error reading resume: {e}"

resume_parser = Tool(
    name="ResumeParser",
    func=resume_parser_tool,
    description="Extracts plain text from a PDF resume file given the file path."
)

def extract_skills_from_job_tool(job_text: str) -> str:
    return job_skill_extractor_chain.invoke({"job_text": job_text})

job_skill_extractor_tool_instance = Tool(
    name="JobSkillExtractor",
    func=extract_skills_from_job_tool,
    description="Extracts required skills from a job description."
)

def skill_gap_analysis_tool(input_data: str) -> str:
    separator = "<JOB>"
    if separator not in input_data:
        return "Error: Missing separator '<JOB>'."
    resume_text, job_text = map(str.strip, input_data.split(separator, 1))
    return skill_gap_analyzer_chain.invoke({"resume_text": resume_text, "job_text": job_text})

skill_gap_analyzer_tool_instance = Tool(
    name="SkillGapAnalyzer",
    func=skill_gap_analysis_tool,
    description="Compares resume and job description to find skill gaps."
)

def interview_question_tool_func(role: str) -> str:
    return interview_question_generator_chain.invoke({"role": role})

interview_question_generator_tool_instance = Tool(
    name="InterviewPrepGenerator",
    func=interview_question_tool_func,
    description="Generates interview questions for a given job role."
)

def strategic_career_plan_tool_func(input_data: str) -> str:
    parts = input_data.split("<SEP>")
    if len(parts) != 4:
        return "Error: Expected 4 parts separated by '<SEP>'."
    resume_skills, job_skills, skill_gaps, job_role = map(str.strip, parts)
    return strategic_career_plan_chain.invoke({
        "resume_skills": resume_skills,
        "job_skills": job_skills,
        "skill_gaps": skill_gaps,
        "job_role": job_role
    })

strategic_career_plan_generator_tool_instance = Tool(
    name="StrategicCareerPlanGenerator",
    func=strategic_career_plan_tool_func,
    description="Generates a career development plan from resume and job data."
)

all_tools = [
    resume_parser,
    job_skill_extractor_tool_instance,
    skill_gap_analyzer_tool_instance,
    interview_question_generator_tool_instance,
    strategic_career_plan_generator_tool_instance
]

# --- JobCoach Agent Initialization (Gemini) ---
gemini_agent_prompt = PromptTemplate.from_template(
    """
    You are JobCoach, an expert AI career mentor, powered by Gemini.
    Your goal is to provide comprehensive career guidance.
    You have access to the following tools:
    {tools}

    Use the following format:
    Question: the user's query
    Thought: reason about what to do, what tools to use, and why.
    Action: the action to take, should be one of [{tool_names}]
    Action Input: the input to the action
    Observation: result of the action
    ... (this Thought/Action/Action Input/Observation can repeat N times)
    Thought: I have gathered all the necessary information and can now formulate the final, structured answer.
    Final Answer:
    =============================
    📈 1. Resume vs Job Description Match (Gemini)
    =============================
    [Your detailed analysis of matching skills and general alignment here.]

    =============================
    🧱 2. Skill Gaps & Personalized Learning Path (Gemini)
    =============================
    [Your identification of skill gaps and specific learning path recommendations here.]

    =============================
    🎯 3. Targeted Interview Questions (Gemini)
    =============================
    [Your list of interview questions here.]

    =============================
    📊 4. Strategic Career Development Plan (Gemini)
    =============================
    [Your comprehensive career development plan here.]

    Begin!

    Question: {input}
    Thought:{agent_scratchpad}
    """
)

gemini_agent = create_react_agent(
    llm=llm_gemini,
    tools=all_tools,
    prompt=gemini_agent_prompt
)

jobcoach_agent_gemini = AgentExecutor(
    agent=gemini_agent,
    tools=all_tools,
    verbose=False,
    handle_parsing_errors=True,
    max_iterations=15,
#    early_stopping_method="generate"
)

# --- Main Execution ---
print("\U0001F4E4 Upload your resume (PDF format)...")
uploaded = files.upload()
file_path = next(iter(uploaded))
resume_text = resume_parser_tool(file_path)

if "Error" in resume_text:
    print(resume_text)
else:
    print("\n\U0001F4C4 Resume parsed successfully!")
    print("\n\U0001F4DD Paste the job description (press Enter twice to finish):")
    job_text = "\n".join(iter(input, ""))
    job_role = input("\n\U0001F50D Job role/title (e.g., 'Data Scientist'): ")

    query = f"""
    Analyze the resume and job description for '{job_role}'.
    Resume:
    {resume_text}

    Job Description:
    {job_text}
    """

    print("\n\U0001F680 Gemini JobCoach:")
    display(Markdown(jobcoach_agent_gemini.invoke({"input": query})["output"]))

print("\n\U0001F4AC Ask career questions (type 'exit' to stop)")
while True:
    user_question = input("\n❓ Your question: ")
    if user_question.lower() in ["exit", "quit"]:
        print("\n👋 Goodbye from JobCoach!")
        break
    display(Markdown(jobcoach_agent_gemini.invoke({"input": user_question})["output"]))


📤 Upload your resume (PDF format)...


Saving Sample_Data_Analyst_Resume.pdf to Sample_Data_Analyst_Resume (17).pdf

📄 Resume parsed successfully!

📝 Paste the job description (press Enter twice to finish):
We are seeking a skilled Data Engineer to design, build, and maintain scalable data pipelines and infrastructure. You will work closely with data scientists, analysts, and software engineers to ensure efficient data flow and accessibility. Key responsibilities include data ingestion, transformation, storage optimization, and ensuring data quality and integrity.


🔍 Job role/title (e.g., 'Data Scientist'): Data Engineer

🚀 Gemini JobCoach:


=============================
📈 1. Resume vs Job Description Match (Gemini)
=============================

Jane Doe's resume demonstrates strong skills in data analysis and visualization, with proficiency in Python, SQL, Tableau, and Excel.  Her experience includes developing dashboards, analyzing customer behavior, and automating data cleaning processes.  However, a direct comparison with the Data Engineer job description reveals a significant mismatch. While her analytical skills are relevant, the job requires expertise in data pipeline design and development, data ingestion, transformation, storage optimization, and ensuring data quality and integrity – areas where her resume shows limited or no experience.  Her current skills are more aligned with a Data Analyst role than a Data Engineer role.  The overlap lies primarily in her proficiency with SQL and Python, which are transferable skills but insufficient for a Data Engineer position.


=============================
🧱 2. Skill Gaps & Personalized Learning Path (Gemini)
=============================

Key skill gaps for Jane Doe to transition to a Data Engineer role include:

* **Data Pipeline Design & Development:**  Experience with tools like Apache Kafka, Apache Airflow, or similar technologies is crucial.
* **Data Ingestion Techniques:**  Familiarity with ingesting data from various sources (databases, APIs, cloud storage).
* **Data Transformation & ETL Processes:**  Practical experience with ETL (Extract, Transform, Load) processes and data transformation techniques.
* **Data Storage Optimization:**  Understanding of different database types (relational, NoSQL) and cloud storage solutions (AWS S3, Azure Blob Storage, GCP Cloud Storage) and their optimization strategies.
* **Data Quality & Integrity:**  Implementing robust data quality checks and ensuring data integrity throughout the pipeline.
* **Cloud Computing Platforms:**  Hands-on experience with at least one major cloud platform (AWS, Azure, or GCP) and relevant services.

**Personalized Learning Path:**

1. **Online Courses:**  Platforms like Coursera, edX, Udacity, and DataCamp offer numerous courses on data engineering, covering the above-mentioned skill gaps.  Focus on courses that include hands-on projects and practical applications.
2. **Cloud Certifications:**  Obtain certifications from AWS (e.g., AWS Certified Data Analytics - Specialty), Azure (e.g., Azure Data Engineer Associate), or GCP (e.g., Google Cloud Professional Data Engineer) to demonstrate proficiency in cloud-based data engineering.
3. **Personal Projects:**  Build personal projects to showcase newly acquired skills.  This could involve creating a data pipeline for a specific dataset, optimizing data storage, or implementing data quality checks.  Contribute to open-source projects to gain experience and build your portfolio.
4. **Networking:**  Attend industry events, connect with data engineers on LinkedIn, and seek mentorship to gain insights and learn from experienced professionals.


=============================
🎯 3. Targeted Interview Questions (Gemini)
=============================

* Tell me about your experience with data analysis and visualization.  Give me a specific example of a project where you used these skills to solve a business problem.
* Describe your experience with SQL.  What are some of the most challenging SQL queries you've written and how did you overcome those challenges?
* What is your understanding of data pipelines?  Have you ever designed or worked with a data pipeline before?  If so, describe your experience.
* What are some common challenges in data engineering, and how would you approach solving them?
* Describe your experience with cloud computing platforms (AWS, Azure, GCP).  What services are you familiar with?
* How do you ensure data quality and integrity in a data pipeline?
* How would you approach designing a data pipeline for a large dataset?  What factors would you consider?
* Describe your experience working collaboratively with data scientists, analysts, and software engineers.
* Tell me about a time you had to troubleshoot a data-related issue.  How did you approach the problem, and what was the outcome?
* Why are you interested in transitioning to a Data Engineer role?  What are your career goals?


=============================
📊 4. Strategic Career Development Plan (Gemini)
=============================

**Phase 1 (3-6 months):** Focus on foundational data engineering concepts and cloud platform skills.  Complete relevant online courses, build small personal projects to demonstrate basic pipeline design and data ingestion, and begin networking with data engineers.

**Phase 2 (6-12 months):**  Deepen expertise in chosen cloud platform.  Complete advanced courses, work on more complex personal projects involving data transformation, storage optimization, and data quality checks.  Pursue relevant cloud certifications.

**Phase 3 (12+ months):**  Actively seek Data Engineer roles.  Highlight your improved skills and projects in your updated resume and portfolio.  Continue networking and seeking mentorship.  Consider contributing to open-source projects to further enhance your skills and build your reputation.  Regularly update your skills and knowledge to stay current with industry trends.


This plan is a guideline; the timeline may vary depending on individual learning pace and opportunities.  Consistent effort and dedication are key to successful career transition.


💬 Ask career questions (type 'exit' to stop)

❓ Your question: what career advice can you share?


**

=============================
📈 1. Resume vs Job Description Match (Gemini)
=============================

The candidate's resume demonstrates relevant skills and experience in software engineering, aligning well with the core requirements of the Senior Software Engineer role. However, there are some significant gaps, primarily in years of experience (requiring 2+ additional years), leadership experience, and specific expertise in microservices architecture and cloud technologies (Azure and GCP).  The candidate's skills in Java, Python, AWS, and Agile methodologies are strong assets.

=============================
🧱 2. Skill Gaps & Personalized Learning Path (Gemini)
=============================

The primary skill gaps are:  Insufficient years of experience, lack of demonstrable leadership experience, limited experience with microservices architecture, and incomplete cloud technology expertise (missing Azure and GCP).  To address these gaps, the candidate should focus on gaining additional experience through projects and leadership opportunities, actively pursuing training and certifications in Azure and GCP, and quantifying accomplishments on their resume to highlight impact.  Networking and mentorship are also crucial for career advancement.

=============================
🎯 3. Targeted Interview Questions (Gemini)
=============================

[See Simulated InterviewPrepGenerator Output above]

=============================
📊 4. Strategic Career Development Plan (Gemini)
=============================

[See Simulated StrategicCareerPlanGenerator Output above]

[See Simulated InterviewPrepGenerator Output above]

=============================
📊 4. Strategic Career Development Plan (Gemini)
=============================

[See Simulated StrategicCareerPlanGenerator Output above]


❓ Your question: exit

👋 Goodbye from JobCoach!


# **Claude JobCoach**

In [ ]:
llm_claude = ChatAnthropic(model="claude-3-5-haiku-latest", temperature=0.2)


# --- Prompt Templates and Chains ---
job_skill_extractor_chain = PromptTemplate.from_template(
    """
    Given this job description:
    {job_text}

    Extract the top technical and soft skills required. List them clearly.
    """
) | llm_claude | StrOutputParser()

resume_skill_extractor_chain = PromptTemplate.from_template(
    """
    Given this resume:
    {resume_text}

    Extract the most prominent technical and soft skills mentioned. List them clearly.
    """
) | llm_claude | StrOutputParser()

skill_gap_analyzer_chain = PromptTemplate.from_template(
    """
    Given this resume:
    {resume_text}

    And this job description:
    {job_text}

    Compare them and identify any specific skill gaps or missing qualifications from the resume perspective,
    relative to the job description's requirements. Be precise.
    """
) | llm_claude | StrOutputParser()

interview_question_generator_chain = PromptTemplate.from_template(
    """
    Create a list of 5-7 targeted behavioral and technical interview questions for the following job role:
    {role}
    """
) | llm_claude | StrOutputParser()

strategic_career_plan_chain = PromptTemplate.from_template(
    """
    Based on the following information, create a strategic career development plan for the user,
    targeting the role of '{job_role}'.

    Resume Skills Summary:
    {resume_skills}

    Job Required Skills Summary:
    {job_skills}

    Identified Skill Gaps:
    {skill_gaps}

    Include short, medium, and long-term goals, recommended resources, and actionable steps.
    """
) | llm_claude | StrOutputParser()

# --- Tools ---
def resume_parser_tool(file_path: str) -> str:
    try:
        with open(file_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            return "".join([page.extract_text() or "" for page in reader.pages]).strip()
    except Exception as e:
        return f"Error reading resume: {e}"

resume_parser = Tool(
    name="ResumeParser",
    func=resume_parser_tool,
    description="Extracts plain text from a PDF resume file given the file path."
)

def extract_skills_from_job_tool(job_text: str) -> str:
    return job_skill_extractor_chain.invoke({"job_text": job_text})

job_skill_extractor_tool_instance = Tool(
    name="JobSkillExtractor",
    func=extract_skills_from_job_tool,
    description="Extracts required skills from a job description."
)

def skill_gap_analysis_tool(input_data: str) -> str:
    separator = "<JOB>"
    if separator not in input_data:
        return "Error: Missing separator '<JOB>'."
    resume_text, job_text = map(str.strip, input_data.split(separator, 1))
    return skill_gap_analyzer_chain.invoke({"resume_text": resume_text, "job_text": job_text})

skill_gap_analyzer_tool_instance = Tool(
    name="SkillGapAnalyzer",
    func=skill_gap_analysis_tool,
    description="Compares resume and job description to find skill gaps."
)

def interview_question_tool_func(role: str) -> str:
    return interview_question_generator_chain.invoke({"role": role})

interview_question_generator_tool_instance = Tool(
    name="InterviewPrepGenerator",
    func=interview_question_tool_func,
    description="Generates interview questions for a given job role."
)

def strategic_career_plan_tool_func(input_data: str) -> str:
    parts = input_data.split("<SEP>")
    if len(parts) != 4:
        return "Error: Expected 4 parts separated by '<SEP>'."
    resume_skills, job_skills, skill_gaps, job_role = map(str.strip, parts)
    return strategic_career_plan_chain.invoke({
        "resume_skills": resume_skills,
        "job_skills": job_skills,
        "skill_gaps": skill_gaps,
        "job_role": job_role
    })

strategic_career_plan_generator_tool_instance = Tool(
    name="StrategicCareerPlanGenerator",
    func=strategic_career_plan_tool_func,
    description="Generates a career development plan from resume and job data."
)

all_tools = [
    resume_parser,
    job_skill_extractor_tool_instance,
    skill_gap_analyzer_tool_instance,
    interview_question_generator_tool_instance,
    strategic_career_plan_generator_tool_instance
]

# --- JobCoach Agent Initialization (Gemini) ---
claude_agent_prompt = PromptTemplate.from_template(
    """
    You are JobCoach, an expert AI career mentor, powered by GPT-4.
    Your goal is to provide comprehensive career guidance.
    You have access to the following tools:
    {tools}

    Use the following format:
    Question: the user's query
    Thought: reason about what to do, what tools to use, and why.
    Action: the action to take, should be one of [{tool_names}]
    Action Input: the input to the action
    Observation: result of the action
    ... (this Thought/Action/Action Input/Observation can repeat N times)
    Thought: I have gathered all the necessary information and can now formulate the final, structured answer.
    Final Answer:
    =============================
    📈 1. Resume vs Job Description Match (Claude)
    =============================
    [Your detailed analysis of matching skills and general alignment here.]

    =============================
    🧱 2. Skill Gaps & Personalized Learning Path (Claude)
    =============================
    [Your identification of skill gaps and specific learning path recommendations here.]

    =============================
    🎯 3. Targeted Interview Questions (Claude)
    =============================
    [Your list of interview questions here.]

    =============================
    📊 4. Strategic Career Development Plan (Claude)
    =============================
    [Your comprehensive career development plan here.]

    Begin!

    Question: {input}
    Thought:{agent_scratchpad}
    """
)

claude_agent = create_react_agent(
    llm=llm_claude,
    tools=all_tools,
    prompt=claude_agent_prompt
)

jobcoach_agent_claude = AgentExecutor(
    agent=claude_agent,
    tools=all_tools,
    verbose=False,
    handle_parsing_errors=True,
    max_iterations=20,
#    early_stopping_method="generate"
)

# --- Main Execution ---
print("\U0001F4E4 Upload your resume (PDF format)...")
uploaded = files.upload()
file_path = next(iter(uploaded))
resume_text = resume_parser_tool(file_path)

if "Error" in resume_text:
    print(resume_text)
else:
    print("\n\U0001F4C4 Resume parsed successfully!")
    print("\n\U0001F4DD Paste the job description (press Enter twice to finish):")
    job_text = "\n".join(iter(input, ""))
    job_role = input("\n\U0001F50D Job role/title (e.g., 'Data Scientist'): ")

    query = f"""
    Analyze the resume and job description for '{job_role}'.
    Resume:
    {resume_text}

    Job Description:
    {job_text}
    """

    print("\n\U0001F680 Claude JobCoach:")
    display(Markdown(jobcoach_agent_claude.invoke({"input": query})["output"]))

print("\n\U0001F4AC Ask career questions (type 'exit' to stop)")
while True:
    user_question = input("\n❓ Your question: ")
    if user_question.lower() in ["exit", "quit"]:
        print("\n👋 Goodbye from JobCoach!")
        break
    display(Markdown(jobcoach_agent_claude.invoke({"input": user_question})["output"]))

📤 Upload your resume (PDF format)...


Saving Sample_Data_Analyst_Resume.pdf to Sample_Data_Analyst_Resume (18).pdf

📄 Resume parsed successfully!

📝 Paste the job description (press Enter twice to finish):
We are seeking a skilled Data Engineer to design, build, and maintain scalable data pipelines and infrastructure. You will work closely with data scientists, analysts, and software engineers to ensure efficient data flow and accessibility. Key responsibilities include data ingestion, transformation, storage optimization, and ensuring data quality and integrity.


🔍 Job role/title (e.g., 'Data Scientist'): Data Engineer

🚀 Claude JobCoach:


=============================
📈 1. Resume vs Job Description Match
=============================
Current Profile: Data Analyst with strong analytical skills
Target Role: Data Engineer
Matching Skills:
- Python programming
- SQL knowledge
- Data analysis background
- Basic data visualization skills

Alignment Percentage: 40-50%
Key Strengths: Analytical mindset, programming fundamentals
Key Gaps: Advanced data engineering technologies

=============================
🧱 2. Skill Gaps & Personalized Learning Path
=============================
Critical Skill Gaps:
1. ETL Processes
2. Big Data Technologies (Hadoop, Spark)
3. Cloud Platform Expertise
4. NoSQL Database Management
5. Data Warehousing
6. Distributed Computing Frameworks

Recommended Learning Path:
- Short-Term (6-12 months):
  • Advanced Python Programming
  • Apache Spark Fundamentals
  • Cloud Platform Certification (AWS/GCP)
  • Advanced SQL and NoSQL Databases

- Resources:
  • DataCamp Data Engineering Track
  • Udacity Data Engineering Nanodegree
  • Coursera IBM Data Engineering Certificate

=============================
🎯 3. Targeted Interview Questions
=============================
Behavioral Questions:
1. Describe a complex data pipeline optimization project
2. How do you ensure data quality in large-scale systems?
3. Explain a challenging cross-team data collaboration

Technical Questions:
1. Compare star and snowflake schema designs
2. Discuss strategies for managing ETL process reliability
3. Demonstrate approach to designing scalable data models

=============================
📊 4. Strategic Career Development Plan
=============================
Career Trajectory:
Junior Data Analyst → Data Analyst → Junior Data Engineer → Data Engineer

Key Milestones:
- Month 1-3: Foundational skill building
- Month 4-6: Advanced technical training
- Month 7-9: Project portfolio development
- Month 10-12: Certification preparation

Estimated Investment:
- Online Courses: $500-$1500
- Certifications: $200-$500
- Learning Resources: $200-$300

Recommended Next Steps:
1. Complete advanced data engineering courses
2. Build GitHub portfolio with data engineering projects
3. Obtain cloud platform certification
4. Network with data engineering professionals

Potential Career Progression:
Junior Data Engineer → Data Engineer → Senior Data Engineer → Data Architecture/Engineering Leadership

Would you like me to provide more detailed guidance on any specific aspect of this career transition plan?


💬 Ask career questions (type 'exit' to stop)

❓ Your question: what career advice can you share?


=============================
📈 1. Career Foundation Principles
=============================
• Self-Assessment: Know your strengths, passions, and core values
• Continuous Learning: Develop a growth mindset
• Networking: Build meaningful professional relationships
• Personal Branding: Consistently communicate your unique professional value

=============================
🧱 2. Skill Development Strategy
=============================
Key Areas to Focus On:
• Technical Skills: Stay current with industry technologies
• Soft Skills: Develop communication, leadership, and adaptability
• Digital Literacy: Understand emerging technologies
• Cross-functional Competencies: Build versatile skill sets

=============================
🎯 3. Career Progression Tactics
=============================
• Set Clear Goals: 
  - Short-term (1-2 years)
  - Mid-term (3-5 years)
  - Long-term (5-10 years)
• Create Actionable Milestones
• Seek Mentorship
• Be Open to Lateral Moves

=============================
📊 4. Professional Growth Roadmap
=============================
Recommended Actions:
• Regular Skills Audit
• Pursue Relevant Certifications
• Attend Industry Conferences
• Engage in Online Learning Platforms
• Build a Robust Professional Network on LinkedIn
• Practice Continuous Feedback Reception

💡 Pro Tips:
- Embrace Change
- Take Calculated Risks
- Invest in Your Personal Development
- Maintain Work-Life Balance

Would you like me to dive deeper into any specific aspect of career development?


❓ Your question: exit

👋 Goodbye from JobCoach!


# User Interface
# **Make sure to open up the link provided **

In [ ]:
# --- Prompt Templates and Chains ---
def create_chains(llm):
    return {
        "job_skill_extractor_chain": PromptTemplate.from_template(
            """
            Given this job description:
            {job_text}

            Extract the top technical and soft skills required. List them clearly.
            """
        ) | llm | StrOutputParser(),

        "resume_skill_extractor_chain": PromptTemplate.from_template(
            """
            Given this resume:
            {resume_text}

            Extract the most prominent technical and soft skills mentioned. List them clearly.
            """
        ) | llm | StrOutputParser(),

        "skill_gap_analyzer_chain": PromptTemplate.from_template(
            """
            Given this resume:
            {resume_text}

            And this job description:
            {job_text}

            Compare them and identify any specific skill gaps or missing qualifications from the resume perspective,
            relative to the job description's requirements. Be precise.
            """
        ) | llm | StrOutputParser(),

        "interview_question_generator_chain": PromptTemplate.from_template(
            """
            Create a list of 5-7 targeted behavioral and technical interview questions for the following job role:
            {role}
            """
        ) | llm | StrOutputParser(),

        "strategic_career_plan_chain": PromptTemplate.from_template(
            """
            Based on the following information, create a strategic career development plan for the user,
            targeting the role of '{job_role}'.

            Resume Skills Summary:
            {resume_skills}

            Job Required Skills Summary:
            {job_skills}

            Identified Skill Gaps:
            {skill_gaps}

            Include short, medium, and long-term goals, recommended resources, and actionable steps.
            """
        ) | llm | StrOutputParser(),
    }

chains = create_chains(llm_gpt4)
chains_gemini = create_chains(llm_gemini)
chains_claude = create_chains(llm_claude)

# --- Tools ---
def resume_parser_tool(file_obj) -> str:
    try:
        file_path = file_obj if isinstance(file_obj, str) else file_obj.get("path", None)
        if not file_path:
            return "❌ Error: Resume file path not found."

        with open(file_path, 'rb') as file:
            reader = PyPDF2.PdfReader(file)
            return "".join([page.extract_text() or "" for page in reader.pages]).strip()
    except Exception as e:
        return f"❌ Error reading resume: {e}"

resume_parser = Tool(
    name="ResumeParser",
    func=resume_parser_tool,
    description="Extracts plain text from a PDF resume file given the file path."
)

def build_tools(chains):
    return [
        resume_parser,
        Tool(name="JobSkillExtractor", func=lambda x: chains["job_skill_extractor_chain"].invoke({"job_text": x}), description="Extracts required skills from a job description."),
        Tool(name="SkillGapAnalyzer", func=lambda x: chains["skill_gap_analyzer_chain"].invoke({"resume_text": x.split("<JOB>")[0].strip(), "job_text": x.split("<JOB>")[1].strip()}) if "<JOB>" in x else "Error: Missing separator '<JOB>'.", description="Compares resume and job description to find skill gaps."),
        Tool(name="InterviewPrepGenerator", func=lambda x: chains["interview_question_generator_chain"].invoke({"role": x}), description="Generates interview questions for a given job role."),
        Tool(name="StrategicCareerPlanGenerator", func=lambda x: strategic_career_plan_tool_func(x, chains), description="Generates a career development plan from resume and job data.")
    ]

def strategic_career_plan_tool_func(input_data: str, chains):
    parts = input_data.split("<SEP>")
    if len(parts) != 4:
        return "Error: Expected 4 parts separated by '<SEP>'."
    resume_skills, job_skills, skill_gaps, job_role = map(str.strip, parts)
    return chains["strategic_career_plan_chain"].invoke({
        "resume_skills": resume_skills,
        "job_skills": job_skills,
        "skill_gaps": skill_gaps,
        "job_role": job_role
    })

all_tools = build_tools(chains)
all_tools_gemini = build_tools(chains_gemini)
all_tools_claude = build_tools(chains_claude)

# --- JobCoach Agent Initialization ---
def build_agent(llm_name: str, llm_model, tools):
    prompt = PromptTemplate(
        input_variables=["input", "tools", "tool_names", "agent_scratchpad"],
        template=f"""
        You are JobCoach, an expert AI career mentor, powered by {llm_name}.
        Your goal is to provide comprehensive career guidance.
        You have access to the following tools:
        {{tools}}

        Use the following format:
        Question: the user's query
        Thought: reason about what to do, what tools to use, and why.
        Action: the action to take, should be one of [{{tool_names}}]
        Action Input: the input to the action
        Observation: result of the action
        ... (this Thought/Action/Action Input/Observation can repeat N times)
        Thought: I have gathered all the necessary information and can now formulate the final, structured answer.
        Final Answer:
        =============================
        📈 1. Resume vs Job Description Match ({llm_name})
        =============================
        [Your detailed analysis of matching skills and general alignment here.]

        =============================
        🩱 2. Skill Gaps & Personalized Learning Path ({llm_name})
        =============================
        [Your identification of skill gaps and specific learning path recommendations here.]

        =============================
        🎯 3. Targeted Interview Questions ({llm_name})
        =============================
        [Your list of interview questions here.]

        =============================
        📊 4. Strategic Career Development Plan ({llm_name})
        =============================
        [Your comprehensive career development plan here.]

        Begin!

        Question: {{input}}
        Thought:{{agent_scratchpad}}
        """
    )
    agent = create_react_agent(llm=llm_model, tools=tools, prompt=prompt)
    return AgentExecutor(agent=agent, tools=tools, verbose=False, handle_parsing_errors=True, max_iterations=15, early_stopping_method="generate")

jobcoach_agent_gpt4 = build_agent("GPT-4", llm_gpt4, all_tools)
jobcoach_agent_gemini = build_agent("Gemini", llm_gemini, all_tools_gemini)
jobcoach_agent_claude = build_agent("Claude", llm_claude, all_tools_claude)

# --- Gradio UI ---
with gr.Blocks() as interface:
    gr.Markdown("""# 💼 JobCoach: Career Development Agent
    Your personal AI career mentor that provides comprehensive career guidance through intelligent conversation.
    """)
    with gr.Row():
        resume_input = gr.File(label="Upload Resume (PDF)")
        job_input = gr.Textbox(label="Paste Job Description", lines=10)
    with gr.Row():
        job_role = gr.Textbox(label="Job Role")
        model_choice = gr.Radio(choices=["GPT-4", "Gemini", "Claude"], label="Select LLM Model", value="GPT-4")
    status_output = gr.Textbox(label="Processing Status", interactive=False)
    jobcoach_output = gr.Markdown(label="JobCoach Output")

    def jobcoach_runner(resume_file, job_text, job_role, model_choice):
        status_output_value = "🔄 Processing..."
        resume_text = resume_parser_tool(resume_file)
        query = f"""
        Analyze the resume and job description for '{job_role}'.
        Resume:
        {resume_text}

        Job Description:
        {job_text}
        """
        try:
            if model_choice == "GPT-4":
                agent = jobcoach_agent_gpt4
            elif model_choice == "Gemini":
                agent = jobcoach_agent_gemini
            elif model_choice == "Claude":
                agent = jobcoach_agent_claude
            result = agent.invoke({"input": query})["output"]
        except Exception as e:
            if model_choice == "Gemini" and "quota" in str(e).lower():
                result = f"⚠️ Gemini quota exceeded. Please switch to GPT-4 or try again later.\n\nError: {str(e)}"
            else:
                result = f"❌ Unexpected error: {str(e)}"
        return "", result

    submit_button = gr.Button("Run JobCoach")
    submit_button.click(jobcoach_runner, inputs=[resume_input, job_input, job_role, model_choice], outputs=[status_output, jobcoach_output], preprocess=False)

    gr.Markdown("""---
    ## 💬 Ask JobCoach a Career Question
    Type a custom career-related question below and receive guidance in real time.
    """)
    user_question = gr.Textbox(label="Your Career Question")
    question_output = gr.Markdown()
    question_submit = gr.Button("Submit Question")

    def handle_question(q):
        try:
            return jobcoach_agent_gpt4.invoke({"input": q})["output"]
        except Exception as e:
            return f"❌ Error: {str(e)}"

    question_submit.click(handle_question, inputs=user_question, outputs=question_output)

    # --- Exit Button ---
    exit_button = gr.Button("Exit JobCoach", variant="stop")
    exit_button.click(fn=lambda: None, js="window.open('', '_self').close()")

interface.launch(show_error=True, inbrowser=True)
display(Javascript('''
    // Close or hide the inline iframe output if rendered in Colab
    let colabOutputFrames = document.querySelectorAll('iframe');
    colabOutputFrames.forEach(frame => frame.parentElement.style.display = 'none');
'''))
# Confirmation
print("✅ JobCoach app initialized. Ready to launch with GPT-4, Gemini, or Claude.")


It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://7a03b3a5178550bd47.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


<IPython.core.display.Javascript object>

✅ JobCoach app initialized. Ready to launch with GPT-4, Gemini, or Claude.
